# Run the SymFormer demo with a temporary public link

This starts the FastAPI backend on a Colab GPU and opens a temporary public Cloudflare Tunnel URL so other devices can reach the browser demo. The URL works only while this notebook's Colab runtime stays connected; it is not permanent hosting.

**Research demo only.** The model performed poorly in validation and is not for diagnosis, screening, or treatment. A detection score is not a probability of disease, and no detections do not mean an image is healthy. The public tunnel has no sign-in; do not upload patient-identifiable or confidential images. Anyone with the link can access the demo while it is running.

## Before you start

1. Open **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run the cells from top to bottom. Google sign-in and permission to mount Drive are required.
3. Keep the matching one-class `epoch_2.pth` checkpoint either directly in `MyDrive/tb-colab/` or in `MyDrive/tb-colab/symformer_oneclass_full_2ep/`. This large checkpoint is deliberately not stored in GitHub.
4. When the last cell prints a `https://*.trycloudflare.com` URL, open it on this or another device. The link changes when you restart the tunnel/runtime.


In [ ]:
from pathlib import Path
import subprocess

PROJECT = Path('/content/tuberculosisdetection')
if not PROJECT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'main',
        'https://github.com/mpk-023/tuberculosisdetection.git', str(PROJECT),
    ], check=True)

REPO = PROJECT / 'Tuberculosis-main'
assert (PROJECT / 'backend' / 'app.py').is_file(), 'Backend source was not found.'
assert (REPO / 'setup.py').is_file(), 'Model source was not found.'
print('Project ready:', PROJECT)

## Install the compatible legacy GPU environment

This compiles the model's custom CUDA operators. On a fresh Colab runtime it can take several minutes. Keep the GPU runtime selected; do not substitute newer MMCV or PyTorch versions.

In [ ]:
import subprocess
from pathlib import Path

MICROMAMBA = Path('/usr/local/bin/micromamba')
ENV = Path('/content/symformer-env')
if not MICROMAMBA.is_file():
    subprocess.run(
        'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local/bin/ --strip-components=1 bin/micromamba',
        shell=True,
        check=True,
    )
if not ENV.is_dir():
    subprocess.run([
        str(MICROMAMBA), 'create', '-y', '-p', str(ENV), '-c', 'conda-forge',
        'python=3.8', 'pip',
    ], check=True)

def run_env(*args):
    subprocess.run([str(MICROMAMBA), 'run', '-p', str(ENV), *map(str, args)], check=True)

run_env('python', '-m', 'pip', 'install', '--upgrade', 'pip<25', 'setuptools<70', 'wheel')
run_env('python', '-m', 'pip', 'install', 'numpy==1.23.5', 'pillow<10')
run_env('python', '-m', 'pip', 'install', 'torch==1.9.0+cu111', 'torchvision==0.10.0+cu111', '-f', 'https://download.pytorch.org/whl/torch_stable.html')
run_env('python', '-m', 'pip', 'install', 'mmcv-full==1.3.12', '-f', 'https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html')
run_env('python', '-m', 'pip', 'install', '-r', str(REPO / 'requirements' / 'runtime.txt'))
run_env('python', '-m', 'pip', 'install', 'timm==0.4.12', 'yapf==0.30.0')
run_env('python', '-m', 'pip', 'install', '-e', str(REPO), '--no-deps', '--no-build-isolation')

check = subprocess.run([
    str(MICROMAMBA), 'run', '-p', str(ENV), 'python', '-c',
    'import sys; print("Python", sys.version); import torch, mmcv; print("torch", torch.__version__, "CUDA runtime", torch.version.cuda, "CUDA available", torch.cuda.is_available(), "mmcv", mmcv.__version__); assert torch.cuda.is_available(), "CUDA is unavailable to this Colab runtime"',
], capture_output=True, text=True)
if check.stdout:
    print(check.stdout)
if check.stderr:
    print(check.stderr)
if check.returncode:
    raise RuntimeError(
        'PyTorch/MMCV/CUDA validation failed in the Python 3.8 environment. '
        'Review the diagnostics above. Confirm Colab is connected to a GPU runtime; '
        'if CUDA is available but mmcv import fails, the matching mmcv-full CUDA wheel '
        'or its compiled operators may not be available for this runtime.'
    )
print('CUDA inference environment ready.')

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/tb-colab')
CHECKPOINT_CANDIDATES = [
    DRIVE_DIR / 'symformer_oneclass_full_2ep' / 'epoch_2.pth',
    DRIVE_DIR / 'epoch_2.pth',
]
EXPECTED_CHECKPOINT = next((path for path in CHECKPOINT_CANDIDATES if path.is_file()), None)
if EXPECTED_CHECKPOINT is None:
    raise FileNotFoundError(
        'One-class checkpoint not found. Checked: ' + ', '.join(map(str, CHECKPOINT_CANDIDATES)) +
        '. Upload the verified epoch_2.pth to MyDrive/tb-colab/ and rerun this cell.'
    )

import os
os.environ['MODEL_CHECKPOINT'] = str(EXPECTED_CHECKPOINT)
os.environ['MODEL_CONFIG'] = str(REPO / 'configs' / 'symformer' / 'symformer_retinanet_r50_oneclass_inference.py')
os.environ['MODEL_DEVICE'] = 'cuda:0'
assert Path(os.environ['MODEL_CONFIG']).is_file(), 'One-class inference config is missing.'
print('Verified checkpoint:', EXPECTED_CHECKPOINT)
print('Inference device:', os.environ['MODEL_DEVICE'])

In [ ]:
import time
import urllib.error
import urllib.request

backend_log_path = Path('/content/symformer-backend.log')
backend_log = backend_log_path.open('w')
backend_process = subprocess.Popen([
    str(MICROMAMBA), 'run', '-p', str(ENV), 'python', '-m', 'uvicorn',
    'backend.app:app', '--app-dir', str(PROJECT), '--host', '0.0.0.0', '--port', '8000',
], cwd=PROJECT, env=os.environ.copy(), stdout=backend_log, stderr=subprocess.STDOUT)

for _ in range(60):
    if backend_process.poll() is not None:
        backend_log.close()
        raise RuntimeError('Backend exited during startup. Log follows:\n' + backend_log_path.read_text())
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            print('Backend health:', response.read().decode())
            break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(2)
else:
    backend_process.terminate()
    raise TimeoutError('Backend did not become healthy. Log follows:\n' + backend_log_path.read_text())

print('Local demo page: http://127.0.0.1:8000/')
print('Backend log:', backend_log_path)

## Refresh an already-running temporary demo (optional)

If you opened this notebook before the latest UI update, stop/interrupt the running tunnel cell first. Run this cell to pull the latest public source and restart only the backend; then run the tunnel cell again. A restarted quick tunnel may give you a new URL. This does not reinstall the model environment or change the checkpoint.


In [ ]:
import subprocess
import time
import urllib.error
import urllib.request

subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
if backend_process.poll() is None:
    backend_process.terminate()
    try:
        backend_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        backend_process.kill()
        backend_process.wait(timeout=10)
if not backend_log.closed:
    backend_log.close()

backend_log_path = Path('/content/symformer-backend.log')
backend_log = backend_log_path.open('w')
backend_process = subprocess.Popen([
    str(MICROMAMBA), 'run', '-p', str(ENV), 'python', '-m', 'uvicorn',
    'backend.app:app', '--app-dir', str(PROJECT), '--host', '0.0.0.0', '--port', '8000',
], cwd=PROJECT, env=os.environ.copy(), stdout=backend_log, stderr=subprocess.STDOUT)

for _ in range(60):
    if backend_process.poll() is not None:
        backend_log.close()
        raise RuntimeError('Backend exited during restart. Log follows:\n' + backend_log_path.read_text())
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            print('Updated backend health:', response.read().decode())
            break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(2)
else:
    backend_process.terminate()
    backend_log.close()
    raise TimeoutError('Updated backend did not become healthy. Log follows:\n' + backend_log_path.read_text())

print('Updated demo frontend is ready at http://127.0.0.1:8000/')


## Open the temporary public link

Run the next cell and wait for the `trycloudflare.com` URL in its output. Leave the cell and Colab runtime running while anyone is using the demo. Stop the cell or disconnect the runtime to take the public link offline.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared.deb
!dpkg -i /content/cloudflared.deb
!cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8000